# Pre-train the multigauge models on CAMELS-US

This notebook builds the pre-trained models that `multigauge` fine-tunes on individual basins:

1. an **LSTM** with a probabilistic (CMAL) output, trained on the 531 CAMELS-US basins used by most deep-learning studies;
2. optionally, the **δHBV hybrid** (differentiable HBV with neural parameter learning).

**Before you start:** *Runtime → Change runtime type → GPU* (a free T4 is enough). Expect about 1 hour for the LSTM and 20–30 minutes for the hybrid. The CAMELS download is about 3.2 GB.

Data: Newman et al. (2015) and Addor et al. (2017), CAMELS v1.2 (NCAR GDEX; downloaded from the Zenodo mirror, record 15529996).

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Install multigauge

In [ ]:
!rm -rf multigauge-hydro && git clone -q https://github.com/ogunbajoadeyinka/multigauge-hydro.git
!pip install -q -e multigauge-hydro

## 2. Download CAMELS-US (forcings + streamflow, and catchment areas)

In [ ]:
%%bash
set -e
mkdir -p /content/camels && cd /content/camels
# CAMELS v1.2 (Newman et al., 2015; Addor et al., 2017), Zenodo mirror of the NCAR GDEX archive
BASE=https://zenodo.org/api/records/15529996/files
[ -s basin_timeseries_v1p2_metForcing_obsFlow.zip ] || wget -nv -O basin_timeseries_v1p2_metForcing_obsFlow.zip $BASE/basin_timeseries_v1p2_metForcing_obsFlow.zip/content
[ -s camels_topo.txt ] || wget -nv -O camels_topo.txt $BASE/camels_topo.txt/content
[ -d basin_dataset_public_v1p2 ] || unzip -q basin_timeseries_v1p2_metForcing_obsFlow.zip
ls

## 3. Load the 531 basins

In [ ]:
import numpy as np, pandas as pd, multigauge as mg
from multigauge.camels import load_camels
ids = open("multigauge-hydro/notebooks/camels_531_basins.txt").read().split()
basins = load_camels("/content/camels", ids, progress=lambda i, n, b: print(f"\r{i}/{n}", end=""))
print("\nloaded", len(basins), "basins")

## 4. Pre-train the LSTM

Standard CAMELS periods (Kratzert et al., 2019): train 1980-10-01 to 1995-09-30, validate 1995-10-01 to 2000-09-30, test 2000-10-01 to 2010-09-30.

In [ ]:
from multigauge.nn import ModelConfig
from multigauge.transfer import pretrain
cfg = ModelConfig.preset("pretrain")   # 128 hidden units, 365-day lookback, CMAL output
print(cfg)
model = pretrain(basins, cfg, ("1980-10-01", "1995-09-30"), ("1995-10-01", "2000-09-30"),
                 progress=lambda e, n, s: print(f"epoch {e}/{n}: median validation NSE {s:.3f}", flush=True))
model.info.update(source="CAMELS-US", description="LSTM pre-trained on 531 CAMELS-US basins (Daymet forcings, CMAL output).")
model.save("multigauge-hydro/src/multigauge/pretrained/camels-us")

## 5. Test-period skill (2000-10-01 to 2010-09-30)

In [ ]:
from multigauge.transfer import period_mask
model.net.to("cuda" if torch.cuda.is_available() else "cpu")
rows = []
for b in basins:
    m = period_mask(b.dates, ("2000-10-01", "2010-09-30"))
    p = model.predict(b)
    rows.append({"basin": b.basin_id, **mg.evaluate_probabilistic(b.q_mm.to_numpy()[m], p[m])})
model.net.to("cpu")
test = pd.DataFrame(rows).set_index("basin")
test.to_csv("multigauge-hydro/src/multigauge/pretrained/camels-us/test_metrics.csv")
print(test[["NSE", "KGE", "CRPS", "Coverage90"]].median().round(3))

*For reference:* published single-forcing CAMELS LSTMs reach a median test NSE of roughly 0.72–0.74. This model uses only climate attributes computed from the forcings (so it can run on any U.S. basin without extra attribute data), so a slightly lower median is expected.

## 6. (Optional) Pre-train the δHBV hybrid

In [ ]:
from multigauge.hybrid import HybridConfig, HybridModel
tr = [period_mask(b.dates, ("1980-10-01", "1995-09-30")) for b in basins]
va = [period_mask(b.dates, ("1995-10-01", "2000-09-30")) for b in basins]
hyb = HybridModel(HybridConfig(epochs=30)).fit(basins, tr, va,
        progress=lambda e, n, s: print(f"epoch {e}/{n}: median validation NSE {s:.3f}", flush=True))
hyb.save("multigauge-hydro/src/multigauge/pretrained/camels-us-dhbv")

# Part 2 — Validate on real basins the model has never seen

Five large basins (all over 2,000 km², so none is among the 531 pre-training basins), in different climates.
For each one the notebook downloads the USGS basin boundary and streamflow, Daymet weather and the NOAA
GHCN-Daily rain gauges inside the basin, then runs the full gauge-value study.

Periods: train water years 2002–2013, validate 2014–2017, test 2018–2023.

In [ ]:
import json, traceback
from pathlib import Path
from multigauge.prepare import prepare_usgs_basin
from multigauge.study import StudyConfig
from multigauge.report import summary_table

SITES = {
    "07326500": "Washita River at Anadarko, OK",
    "05453100": "Iowa River at Marengo, IA",
    "02347500": "Flint River, GA (USGS 02347500)",
    "08167000": "Guadalupe River at Comfort, TX",
    "01643000": "Monocacy River at Jug Bridge, MD",
}
START, END = "2000-10-01", "2023-09-30"
PERIODS = dict(train_period=("2001-10-01", "2013-09-30"), validation_period=("2013-10-01", "2017-09-30"),
               test_period=("2017-10-01", "2023-09-30"))
base = mg.load_pretrained("multigauge-hydro/src/multigauge/pretrained/camels-us")
try:
    hybrid = hyb            # from section 6, if it was run
except NameError:
    hybrid = None
OUT = Path("/content/validation"); OUT.mkdir(exist_ok=True)

In [ ]:
results, data_by_site = {}, {}
for site, label in SITES.items():
    try:
        d = prepare_usgs_basin(site, START, END, OUT / site / "data", name=label)
        cfg = StudyConfig(n_seeds=3, finetune_epochs=20, run_ablation=False, **PERIODS)
        res = mg.run_study(d, base, cfg, hybrid=hybrid,
                           progress=lambda f, m: print(f"\r[{site}] {int(f*100):3d}% {m[:70]:<70}", end=""))
        print()
        out = res.save(OUT / site / "results")
        mg.write_report(res, d, out / "report.html")
        results[site], data_by_site[site] = res, d
        print(summary_table(res).round(3)[["NSE", "KGE", "CRPS"]].to_string(), "\n")
    except Exception as exc:
        print(f"\n[{site}] FAILED: {exc}")
        traceback.print_exc()

## Washita extras: sub-basins and forecasts

Splits the Washita into 4 distance-band zones (with their own Daymet weather), compares the lumped model with
sub-basin routing, and tests 1–7 day forecast skill.

In [ ]:
import numpy as np
from multigauge.routing import SubbasinSet, auto_zones, run_routed_study
from multigauge.forecast import ForecastConfig, ForecastModel
from multigauge.download import daymet_basin
from multigauge.study import make_periods
from shapely.geometry import mapping

site = "07326500"
if site in data_by_site:
    d = data_by_site[site]
    zdir = OUT / site / "data" / "subbasins"; zdir.mkdir(exist_ok=True)
    zones = auto_zones(d.boundary, d.outlet, 4)
    feats = []
    for i, z in enumerate(zones):
        zid = f"Z{i+1}"
        daymet_basin(z, START, END, n_points=9).to_csv(zdir / f"{zid}.csv", date_format="%Y-%m-%d")
        feats.append({"type": "Feature", "geometry": mapping(z), "properties": {"id": zid, "downstream": None if i == 0 else f"Z{i}"}})
    (zdir / "subbasins.geojson").write_text(json.dumps({"type": "FeatureCollection", "features": feats}))
    subs = SubbasinSet.from_folder(zdir)
    periods = make_periods(d.dates, StudyConfig(**PERIODS), base.cfg.lookback)
    routed = run_routed_study(d, base, subs, periods, routing="gamma", epochs=15, gauge_ablation=True)
    print(routed.summary.xs("test", level="period")[["NSE", "KGE"]].round(3))
    routed.summary.to_csv(OUT / site / "results" / "subbasin_models.csv")
    routed.contributions.to_csv(OUT / site / "results" / "subbasin_contributions.csv")

    b = d.to_basin()
    m = {k: np.asarray((d.dates >= a) & (d.dates <= e)) for k, (a, e) in periods.items()}
    ft = mg.finetune(base, b, m["train"], m["validation"], epochs=20)
    fm = ForecastModel(ft, ForecastConfig(epochs=15)).fit(b, m["train"], m["validation"])
    iss = np.where(m["test"])[0]; iss = iss[iss < len(b.dates) - 8]
    skill = fm.skill_by_lead(b, iss)
    skill.to_csv(OUT / site / "results" / "forecast_skill.csv", index=False)
    print(skill.round(3).to_string(index=False))
    mg.write_report(results[site], d, OUT / site / "results" / "report.html", routed=routed, forecast_skill=skill)

## Optional: Washita with your Oklahoma Mesonet gauges

Upload `rainfall.csv` and `stations.csv` (same format as multigauge's examples) for the Mesonet stations.
The study then reruns with Mesonet instead of NOAA gauges. **These files stay in this Colab session; check the
Mesonet terms before sharing any results that include their data.**

In [ ]:
from google.colab import files
up = files.upload()   # choose rainfall.csv and stations.csv, or press Cancel to skip
if {"rainfall.csv", "stations.csv"} <= set(up):
    import shutil
    mdir = OUT / "07326500_mesonet" / "data"; mdir.mkdir(parents=True, exist_ok=True)
    for f in ("forcings.csv", "streamflow.csv", "boundary.geojson", "basin.json"):
        shutil.copy(OUT / "07326500" / "data" / f, mdir / f)
    for f in ("rainfall.csv", "stations.csv"):
        (mdir / f).write_bytes(up[f])
    dm = mg.BasinData.from_folder(mdir, name="Washita River at Anadarko, OK (Mesonet gauges)")
    rm = mg.run_study(dm, base, StudyConfig(n_seeds=3, finetune_epochs=20, run_ablation=True, **PERIODS), hybrid=hybrid)
    out = rm.save(OUT / "07326500_mesonet" / "results"); mg.write_report(rm, dm, out / "report.html")
    results["07326500_mesonet"] = rm
    print(summary_table(rm).round(3)[["NSE", "KGE", "CRPS"]].to_string())

## Summary across basins

In [ ]:
rows = []
for site, res in results.items():
    s = res.summary.xs("test", level="period")
    gv = res.gauge_value[res.gauge_value["period"] == "test"].set_index("gauge_input") if res.gauge_value is not None else None
    r = {"site": site, "basin": res.basin_name, "area_km2": round(res.area_km2), "gauges": len(res.station_summary)}
    for mname in ["gr4j", "local", "pretrained", "finetuned", "ft_voronoi", "ft_disaggregated", "dhbv", "ensemble"]:
        if mname in s.index:
            r[f"NSE_{mname}"] = round(s.loc[mname, "NSE"], 3)
    if gv is not None:
        r["best_gauge_input"] = gv["NSE_gain"].idxmax()
        r["NSE_gain_from_gauges"] = round(gv["NSE_gain"].max(), 3)
        r["CRPS_reduction_pct"] = round(gv.loc[gv["NSE_gain"].idxmax(), "CRPS_reduction_pct"], 1)
    rows.append(r)
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "summary.csv", index=False)
summary

## Download everything

Attach `multigauge_results.zip` in your chat with Claude. It contains the trained models and the validation results.

In [ ]:
!cd /content && rm -f multigauge_results.zip && cp -r multigauge-hydro/src/multigauge/pretrained/camels-us* validation/ 2>/dev/null; zip -qr multigauge_results.zip validation; ls -la multigauge_results.zip
from google.colab import files
files.download("/content/multigauge_results.zip")